# 🧠 CerebroSpheroid V4: Cloud TPU JAX High-Throughput Engine
### 128 GB HBM • 8 TPU v3 Cores • Pure JAX / PJRT Native • 0ms Compilation Delay

### **Architecture & Capabilities in V4-Small:**
1. **Modern Llama 32,000 Vocabulary**: Byte-fallback BPE with atomic single-digit arithmetic (`0`-`9`) and zero OOV tokens (127M total parameters).
2. **Elastic Sliding Context Window**: High-density dynamic reasoning buckets without padding traps.
3. **Curriculum Scheduled Sampling**: Linearly transitions from teacher forcing ($p=1.0$) to closed-loop internal deliberation ($p=0.0$) over 5,000 steps.
4. **Riemannian SpheroidOptimizer**: Polar Quintic Newton-Schulz optimizer with single-moment memory and contractive spectral bounds ($\rho \le 1.0$).
5. **Held-Out Validation & Direct Cloud Sync**: Continuously evaluates generalization loss on OpenHermes-2.5 and GSM8K and mirrors checkpoints chunk-by-chunk to `bioflo.in` without TPU stalls.

---
### ⚡ Kaggle Settings:
- **Accelerator**: Select **TPU VM v3-8**
- **Internet**: Toggle **Internet ON**
- **Persistence**: Variables and Files


## Step 1: Verify Kaggle TPU Hardware & JAX PJRT Environment

In [ ]:
import jax
import jax.numpy as jnp

print(f"JAX Version      : {jax.__version__}")
print(f"JAX Default Dev  : {jax.default_backend().upper()}")
devices = jax.devices()
print(f"Detected Devices : {devices}")
print(f"Active Core Count: {len(devices)}")
print(f"Device Kind      : {devices[0].device_kind}")

is_tpu = any("TPU" in d.device_kind.upper() for d in devices)
assert is_tpu, "⚠️ No TPU detected! In the Kaggle notebook settings (right panel), set Accelerator to 'TPU VM v3-8'!"
print(f"\n✅ Successfully connected to Cloud TPU VM! ({len(devices)} cores active)")


## Step 2: Clone / Pull Latest Repository & Dependencies

In [ ]:
import os
if not os.path.exists("/kaggle/working/reasoning_agent"):
    !git clone https://github.com/AjitKumar-git/reasoning_agent.git /kaggle/working/reasoning_agent
%cd /kaggle/working/reasoning_agent
!git fetch origin main
!git reset --hard origin/main
!pip install -q transformers datasets sentencepiece optax torch


## Step 3: Checkpoint Ingestion & Resumption (Automatic)

In [ ]:
import os, glob, shutil
os.makedirs("/kaggle/working/checkpoints", exist_ok=True)
# 1. Check mounted inputs under /kaggle/input
if os.path.exists("/kaggle/input"):
    for root, dirs, files in os.walk("/kaggle/input"):
        for f in files:
            full_p = os.path.join(root, f)
            if f.endswith(".zip") and "cerebro_spheroid" in f:
                print(f"📦 Unzipping checkpoint archive from input: {full_p}")
                !unzip -q -o "{full_p}" -d /kaggle/working/checkpoints/
            elif f.endswith(".pt") and "cerebro_spheroid" in f:
                shutil.copy2(full_p, os.path.join("/kaggle/working/checkpoints", f))
                print(f"📥 Found raw checkpoint: {f} -> copied to /kaggle/working/checkpoints/")

ckpts = glob.glob("/kaggle/working/checkpoints/*.pt")
if ckpts:
    print(f"✅ Active checkpoints ready for auto-resume ({len(ckpts)} files found):")
    for c in ckpts:
        print(f"   • {os.path.basename(c)} ({os.path.getsize(c) / (1024**2):.1f} MB)")
else:
    print("🆕 No existing checkpoints found in inputs. Will auto-sync latest state from bioflo.in or start clean!")


## Step 4: Launch CerebroSpheroid V4 JAX Cloud TPU Training
- **Engine**: Pure JAX native execution with XLA fusion
- **Context**: Elastic Sliding Window (1,024 tokens budget)
- **Curriculum**: Scheduled sampling over 5,000 steps
- **Validation**: Frozen held-out OpenHermes + GSM8K eval every 200 steps
- **Time Safety**: `--max_hours 8.5` safely exits before Kaggle's 9.0-hr hard timeout
- **Cloud Sync**: 0ms TPU pause asynchronous mirroring to bioflo.in

In [ ]:
!python3 training/train_v4_jax.py     --ckpt_dir /kaggle/working/checkpoints     --resume_target best     --dataset_source balanced_mix     --elastic_window     --total_window 1024     --batch_size 8     --grad_accum 2     --lr 0.0005     --deliberation_mode scheduled     --schedule_steps 20000     --min_sampling_p 0.35     --repetition_penalty 1.25     --eval_interval 200     --thalamic_anchor_weight 0.1     --max_steps 120000     --max_hours 8.5     --save_every 100     --log_every 10     --cloud_sync


## Step 5: Archive Checkpoints & Verify Bioflo Cloud Sync

In [ ]:
import os, glob, json
from src.cloud_sync import check_remote_manifest

!mkdir -p /kaggle/working/checkpoints_tpu
!cp -r /kaggle/working/checkpoints/* /kaggle/working/checkpoints_tpu/ 2>/dev/null || true
!zip -r /kaggle/working/cerebro_spheroid_v4_jax_checkpoints.zip /kaggle/working/checkpoints_tpu
!ls -lh /kaggle/working/*.zip

print("\n🌐 Remote bioflo.in manifest:")
print(json.dumps(check_remote_manifest(), indent=2))
print("\n✅ Checkpoints successfully archived and mirrored!")
